# Phishing Email Detection with Explainable AI (Kaggle GPU Runner)

This notebook runs the **full production pipeline** on Kaggle (GPU) and replaces the
old Colab flow (`notebooks/colab_run.ipynb`, kept in the repo for reference).
Run the cells **top to bottom, in order**. Nothing in this notebook is executed
locally and no metric below is claimed until the cell that produces it has run.

## Before you press Run All — Kaggle setup (do this first)

1. **Accelerator: GPU.** Right-hand panel Settings -> Accelerator -> **GPU**
   (T4 or P100). Cell 1 stops with a warning if no GPU is found.
2. **Internet: ON.** Settings -> Internet -> **ON**. Needed for `apt-get`,
   `pip install`, and the dataset download in Cell 4.
3. **Attach the repo as a dataset.** Kaggle Notebook -> Add Input -> Upload or
   "New Dataset": upload a zip/folder of this repo (it must contain
   `scripts/run_train.py` and `requirements-kaggle.txt`). It appears read-only
   under `/kaggle/input/<dataset-name>/`. Cell 3 copies it to writable
   `/kaggle/working/`.
4. **(Optional) Attach the data + checkpoint.**
   - Data: if `data/Phishing_Email.csv` is already inside the repo dataset, Cell 4
     uses it. Otherwise attach the CSV as a second dataset, or let Cell 4 download
     it from the Hugging Face mirror (Internet must be ON).
   - Checkpoint (to skip training): attach a dataset containing `best_model.pt`.
     Cell 3 copies it to `/kaggle/working/results/best_model.pt`. Training
     (Cell 6) then still runs unless you skip it — see the note on Cell 6.
5. **Save output.** After the run, use **Save Version** (output saved) so
   `/kaggle/working/results/` persists. To reuse the checkpoint later, attach
   that saved output as an input dataset in the next notebook (Add Input ->
   search the previous notebook run), or download `results/best_model.pt` and
   upload it via Datasets -> New Dataset. See the last markdown cell.

## Precision first (user priority, 2026-11-19)

**Phishing-class PRECISION is the primary metric** (minimise false positives /
low FPR). F1, accuracy and recall are reported everywhere but never used alone
for model selection. That means:

- Any decision threshold is tuned on the **validation** split only, never on test
  (see the note on the multimodal-predict cell).
- The complexity-report cells judge the verdict on the **precision** gain first
  (F1 is the secondary statement), and the recommended deployment model is the
  highest-precision model subject to recall >= 0.85 on validation aggregates.

## What runs, in order

1. GPU check | 2. System tools + `requirements-kaggle.txt` | 3. Path setup (repo
   -> `/kaggle/working`) | 4. Data download / verify + splits | 5. Optional tuning
   | 6. Training (resume into `/kaggle/working/results/best_model.pt`)
   | 7. Explain + faithfulness + ablation | 8. Multimodal samples, eval set and
   prediction | 9. Section H baselines (H8) + complexity report
   | 10. Consolidated report (`make_report.py`) | 11. Output-file listing.

## Status notes (honest, not aspirational)

- **Section E has landed in this repo snapshot.** `scripts/make_multimodal_eval_set.py`
  and `scripts/run_multimodal_eval.py` both exist (verified `--help` 2026-11-19:
  eval set `--out_dir` / `--seed` / `--debug`; evaluator `--eval_dir` /
  `--checkpoint` / `--output_dir` / `--fusion weighted|max` / `--models
  tfidf_lr bilstm roberta` / `--recall_floor`). The evaluator cells still guard
  with an existence check so an older dataset snapshot prints `PENDING` instead
  of failing. Do **not** rename either script.
- Every number in `results/` is **NOT RUN YET** until its cell has actually
  executed on Kaggle. Nothing here re-uses a local result.
- The old Colab notebook (`notebooks/colab_run.ipynb`) is untouched.


In [ ]:
# Cell 1: Check GPU (Kaggle: Settings -> Accelerator -> GPU, T4/P100)
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Training will be very slow on CPU; "
          "enable a GPU accelerator (Settings -> Accelerator -> GPU) and re-run from here.")

In [ ]:
# Cell 2: System tools + Python deps.
# IMPORTANT: Internet must be ON in the Kaggle notebook Settings (right-hand panel),
# otherwise apt-get and pip below will fail.
# tesseract-ocr -> pytesseract (OCR); libzbar0 -> pyzbar (QR decoding).
!apt-get update -qq && apt-get install -y -qq tesseract-ocr libzbar0
!which tesseract

# requirements-kaggle.txt lives in the attached repo dataset under /kaggle/input
# (the repo is copied to writable /kaggle/working in Cell 3). It lists MINIMUMS
# only and deliberately does NOT reinstall torch: Kaggle already ships a CUDA
# torch build.
import glob
_req = glob.glob('/kaggle/input/**/requirements-kaggle.txt', recursive=True)
assert _req, "requirements-kaggle.txt not found under /kaggle/input - attach the repo dataset first."
REQ = _req[0]
print("Using:", REQ)
!pip install -q -r $REQ
print("Dependencies installed.")
import pytesseract, pyzbar  # noqa: F401  (fails loudly here if the apt step above was skipped)
print("pytesseract + pyzbar import OK.")

In [ ]:
# Cell 3: Path setup - copy the repo dataset into writable /kaggle/working
# (/kaggle/input is read-only, so all outputs go under /kaggle/working/results/).
import glob
import os
import shutil
from pathlib import Path

def _find_repo_root():
    # Repo root = directory containing scripts/run_train.py; check each dataset
    # root, then one level down (in case the dataset wraps the repo in a folder).
    for base in sorted(glob.glob('/kaggle/input/*')):
        if os.path.exists(os.path.join(base, 'scripts', 'run_train.py')):
            return base
        for sub in sorted(glob.glob(os.path.join(base, '*'))):
            if os.path.isdir(sub) and os.path.exists(os.path.join(sub, 'scripts', 'run_train.py')):
                return sub
    return None

REPO_SRC = _find_repo_root()
assert REPO_SRC is not None, "Repo not found under /kaggle/input - attach the dataset that contains the repo."
WORKING = Path('/kaggle/working')
print("Repo dataset:", REPO_SRC)
shutil.copytree(REPO_SRC, WORKING, dirs_exist_ok=True)

# Optional pre-trained checkpoint shipped as its own dataset (or inside the repo
# dataset): copy it to /kaggle/working/results/best_model.pt. Training (Cell 6)
# and the explain/faithfulness cells read it from there; run_multimodal_predict.py
# would also auto-detect it under /kaggle/input, but an explicit copy keeps every
# script pointed at the same file.
for ckpt in sorted(glob.glob('/kaggle/input/**/best_model.pt', recursive=True)):
    dest = WORKING / 'results' / 'best_model.pt'
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        shutil.copy2(ckpt, dest)
        print("Checkpoint copied from dataset:", ckpt)
    else:
        print("Checkpoint already present at:", dest)
    break
else:
    print("No best_model.pt found in attached datasets; Cell 6 will train one.")

%cd /kaggle/working
# Editable install WITHOUT deps: only registers the src/ package. Plain
# `pip install -e .` would reinstall torch/transformers from pyproject.toml,
# which Kaggle already provides - hence --no-deps.
!pip install -q -e . --no-deps
!ls
print("Working directory is now /kaggle/working (repo copy).")

In [ ]:
# Cell 4: Dataset - use the shipped CSV, else an attached CSV, else download + verify.
# scripts/download_data.py downloads from the Hugging Face mirror and STRICTLY
# verifies schema (columns 'Email Text'/'Email Type'), labels and row count
# (~18,650 rows). It skips the download when the file already exists but still
# verifies it, so running it here is always the verification step.
import glob
import shutil
from pathlib import Path
if not Path('data/Phishing_Email.csv').exists():
    found = glob.glob('/kaggle/input/**/Phishing_Email.csv', recursive=True)
    if found:
        Path('data').mkdir(exist_ok=True)
        shutil.copy2(found[0], 'data/Phishing_Email.csv')
        print("CSV copied from attached dataset:", found[0])
    else:
        print("No CSV attached; downloading from the Hugging Face mirror (needs Internet ON)...")
        !python scripts/download_data.py --output_path data/Phishing_Email.csv
else:
    print("data/Phishing_Email.csv already present; verifying only.")
    !python scripts/download_data.py --output_path data/Phishing_Email.csv

# Leakage-free stratified splits -> results/splits.json (train 11215 / val 2803 / test 3495).
!python scripts/make_splits.py --data_path data/Phishing_Email.csv --output_dir results --seed 42

In [ ]:
# Cell 5 (OPTIONAL): Random-search hyperparameter tuning.
# Skips itself when results/tuning/best_params.json already exists (e.g. shipped in
# the dataset). Delete that file to force a fresh tuning run. Tuning is scored on
# VALIDATION only, never on test.
import os
if not os.path.exists('results/tuning/best_params.json'):
    !python scripts/run_tune.py --data_path data/Phishing_Email.csv --output_dir results --n_iterations 10 --max_length 256 --subsample_train 4000 --seed 42
else:
    print("results/tuning/best_params.json already exists; skipping tuning.")

In [ ]:
# Cell 6: Train RoBERTa. Checkpoint resumes into /kaggle/working/results/best_model.pt
# (--resume picks up results/checkpoints/latest_checkpoint.pt after an interruption;
# --ckpt_dir keeps the large intermediate checkpoints under results/checkpoints/).
# To SKIP training and reuse an attached checkpoint instead, skip this cell - but
# only if Cell 3 copied best_model.pt (check with !ls -lh results/best_model.pt).
import os
if os.path.exists('results/tuning/best_params.json'):
    !python scripts/run_train.py --data_path data/Phishing_Email.csv --output_dir /kaggle/working/results --params results/tuning/best_params.json --ckpt_dir /kaggle/working/results/checkpoints --resume --max_length 512 --seed 42
else:
    !python scripts/run_train.py --data_path data/Phishing_Email.csv --output_dir /kaggle/working/results --ckpt_dir /kaggle/working/results/checkpoints --resume --max_length 512 --seed 42
!ls -lh /kaggle/working/results/best_model.pt

## Explainability: LIME, Transformers-Interpret, LITA, faithfulness, ablation

Cell 7 runs the three explainability stages against the checkpoint from Cell 6.
All three read the **same** `results/splits.json` as training, so the explained
instances are test-split emails. No threshold is tuned here; faithfulness and
ablation report comprehensiveness / sufficiency, they do not re-tune the model.


In [ ]:
# Cell 7: Explain + faithfulness + ablation (all against /kaggle/working/results/best_model.pt)
# LIME + Transformers-Interpret + LITA local explanations for 10 test emails.
!python scripts/run_explain.py --checkpoint_path /kaggle/working/results/best_model.pt --output_dir /kaggle/working/results/explanations --num_samples 10 --seed 42
# Faithfulness (comprehensiveness / sufficiency vs a random baseline) on 100 test emails.
!python scripts/run_faithfulness.py --model_path /kaggle/working/results/best_model.pt --output_dir /kaggle/working/results --sample_size 100 --max_length 256 --seed 42
# LITA alpha ablation (k=3) on 50 test emails.
!python scripts/run_ablation.py --model_path /kaggle/working/results/best_model.pt --output_dir /kaggle/working/results --sample_size 50 --k 3 --max_length 256 --seed 42

## Multimodal: image / PDF phishing (OCR + QR + PDF text)

Cell 8 regenerates the 6 synthetic `.eml` samples, scores them with the trained
checkpoint, and measures LITA faithfulness on them.

**Precision-first note on the threshold:** `run_multimodal_predict.py` is called
with `--fusion weighted` (the precision-first fusion; `max` is the recall-heavy
alternative) and **no `--threshold` flag**, so it uses its 0.5 default. That 0.5
is a placeholder: any tuned threshold must be fitted on the **validation** split
only, never on test and never on these 6 samples. If you tune one, pass it
explicitly as `--threshold <value>` and record where it came from.

**Section E (landed):** `scripts/make_multimodal_eval_set.py` and
`scripts/run_multimodal_eval.py` both exist in this snapshot (verified `--help`
2026-11-19). The evaluator cells still guard with an existence check so an
older dataset snapshot prints `PENDING` instead of failing; do not rename the
scripts.


In [ ]:
# Cell 8: Multimodal samples + prediction + multimodal faithfulness + Section E eval set
# Regenerate the 6 synthetic .eml samples (image-only phish, button image, QR phish,
# text PDF, scanned PDF, clean benign) - needs fitz/PyMuPDF + qrcode + Pillow.
!python scripts/make_multimodal_samples.py
# Score the 6 samples. --fusion weighted is the precision-first default;
# --threshold is intentionally NOT passed (see the markdown note above).
!python scripts/run_multimodal_predict.py samples/multimodal --checkpoint /kaggle/working/results/best_model.pt --output_dir /kaggle/working/results/multimodal --fusion weighted --max_samples 6
# LITA faithfulness (top-k=5) on the 6 multimodal samples.
!python scripts/run_multimodal_faithfulness.py --samples_dir samples/multimodal --checkpoint /kaggle/working/results/best_model.pt --output_json /kaggle/working/results/multimodal_faithfulness.json

# --- Section E eval set + evaluator (both exist in this snapshot; guarded anyway) ---
import os
if os.path.exists('scripts/make_multimodal_eval_set.py'):
    !python scripts/make_multimodal_eval_set.py --out_dir /kaggle/working/data/multimodal_eval --seed 42
else:
    print("PENDING (Section E, not in this repo snapshot): scripts/make_multimodal_eval_set.py - skipped.")
if os.path.exists('scripts/run_multimodal_eval.py'):
    !python scripts/run_multimodal_eval.py --eval_dir /kaggle/working/data/multimodal_eval --checkpoint /kaggle/working/results/best_model.pt --output_dir /kaggle/working/results/multimodal_eval --fusion weighted --models tfidf_lr bilstm roberta
else:
    print("PENDING (Section E, not in this repo snapshot): scripts/run_multimodal_eval.py - skipped.")

## Section H baselines + complexity report (H8)

Cells 9-15 run the baseline ladder in this order: cheap baselines (dummy,
TF-IDF+LR, TF-IDF char-SVM) -> BiLSTM -> optional 1D-CNN / DistilBERT ->
RoBERTa checkpoint eval -> error analysis + robustness -> faithfulness ->
Section E multimodal eval -> complexity report.

**Two things to know before running them:**

1. **Precision first.** Every run ranks models by phishing-class **precision**
   (recall floor 0.85 on validation aggregates); the complexity report judges
   its verdict on the precision gain, with F1 as the secondary statement.
   Thresholds are fitted on validation only.
2. **Each `run_baselines.py` call rewrites `results/baselines/baseline_results.json`.**
   So the complexity report (Cell 15) only sees the models from the **most
   recent** baseline invocation, not the accumulated ladder. The cells below are
   the per-stage runs from the Section H spec; if you want one combined table in
   the report, run the single consolidated command in the optional cell before
   Cell 15 (it re-trains the ladder in one invocation, so it costs the most time
   - only do it when the per-stage runs have already validated the pipeline).


In [ ]:
# Cell 9 (H8): Cheap baselines - dummy, TF-IDF+LR, TF-IDF char-SVM, 3 seeds (42/1/7).
# Output auto-redirects to /kaggle/working/results/baselines on Kaggle; the
# --output_dir below states the same path explicitly.
!python scripts/run_baselines.py --models dummy tfidf_lr tfidf_svm --seeds 42 1 7 --output_dir /kaggle/working/results/baselines

In [ ]:
# Cell 10 (H8): BiLSTM baseline (from-scratch embedding, 8 epochs), 3 seeds.
!python scripts/run_baselines.py --models bilstm --seeds 42 1 7 --epochs 8 --emb_dim 200 --hidden 128 --layers 2 --max_len 256 --output_dir /kaggle/working/results/baselines

In [ ]:
# Cell 11 (H8, OPTIONAL): 1D-CNN + DistilBERT, 1 seed.
# Slow, and DistilBERT downloads weights when they are not cached - skip this cell
# if you only need the cheap + BiLSTM + RoBERTa comparison. --with_cnn /
# --with_distilbert are the Section H flags for these two models (they are also
# named in --models here, which is equivalent; the flags are kept as specified).
!python scripts/run_baselines.py --models cnn distilbert --seeds 42 --with_cnn --with_distilbert --output_dir /kaggle/working/results/baselines

In [ ]:
# Cell 12 (H8): RoBERTa checkpoint eval - scores the Cell 6 checkpoint on the test
# split. Never retrains. Fails honestly (NOT RUN YET) if best_model.pt is missing.
!python scripts/run_baselines.py --models roberta_eval --checkpoint /kaggle/working/results/best_model.pt --output_dir /kaggle/working/results/baselines

In [ ]:
# Cell 13 (H8): Error analysis + robustness (3 perturbations: truncated-128,
# URLs-removed, typo/synonym) on the fitted first-seed TF-IDF+LR and BiLSTM models.
# Writes results/baselines/error_analysis.csv. (Rewrites baseline_results.json
# with just these two models - see the markdown note above Cell 9.)
!python scripts/run_baselines.py --models tfidf_lr bilstm --seeds 42 --analysis --robustness --output_dir /kaggle/working/results/baselines

In [ ]:
# Cell 14 (H8): Faithfulness (comprehensiveness / sufficiency at k in {1,3,5})
# for TF-IDF+LR (native coefficients + LIME) and BiLSTM (LIME + Captum IG),
# 100 samples. (Rewrites baseline_results.json with just these two models -
# see the markdown note above Cell 9.)
!python scripts/run_baselines.py --models tfidf_lr bilstm --seeds 42 --faithfulness --sample_size 100 --output_dir /kaggle/working/results/baselines

In [ ]:
# Cell 15 (H8, OPTIONAL consolidated run): one invocation covering the whole ladder
# with analysis + robustness + faithfulness, so that baseline_results.json - and
# therefore the Cell 16 complexity report - holds ONE combined table.
# It re-trains everything, so only run it after the per-stage cells above have
# validated the pipeline. Uncomment the command below to run it.
# !python scripts/run_baselines.py --models dummy tfidf_lr tfidf_svm bilstm roberta_eval --seeds 42 1 7 --epochs 8 --emb_dim 200 --hidden 128 --layers 2 --max_len 256 --checkpoint /kaggle/working/results/best_model.pt --analysis --robustness --faithfulness --sample_size 100 --output_dir /kaggle/working/results/baselines

In [ ]:
# Cell 16 (H8): Multimodal eval (Section E) across baseline models, then the
# complexity report. The report reads results/baselines/baseline_results.json
# (most recent baseline invocation only - see the note above Cell 9) and writes
# the precision-first verdict + plots into /kaggle/working/results/.
import os
if os.path.exists('scripts/run_multimodal_eval.py'):
    !python scripts/run_multimodal_eval.py --eval_dir /kaggle/working/data/multimodal_eval --checkpoint /kaggle/working/results/best_model.pt --output_dir /kaggle/working/results/multimodal_eval --fusion weighted --models tfidf_lr bilstm roberta
else:
    print("PENDING (Section E, not in this repo snapshot): scripts/run_multimodal_eval.py - skipped.")
!python scripts/run_complexity_report.py --results_dir /kaggle/working/results --output_dir /kaggle/working/results

In [ ]:
# Cell 17: Consolidated report (markdown + JSON) over everything in /kaggle/working/results.
!python scripts/make_report.py --results_dir /kaggle/working/results
print("Full pipeline complete! Consolidated RESULTS.md:")
!cat /kaggle/working/results/RESULTS.md

## Saving the checkpoint for reuse + what to download

**Save the checkpoint as a Kaggle Dataset for reuse (do this before closing the
notebook):**

- **Option A (recommended):** Save Version with output saved, then in the next
  notebook use Add Input and attach that saved run output as an input dataset.
  The checkpoint is then found at `/kaggle/input/<that-output>/best_model.pt`
  (Cell 3 copies it back into `results/` automatically).
- **Option B:** Download `results/best_model.pt` from the Output panel (see the
  listing in the next cell) and upload it via Kaggle Datasets -> New Dataset.
  Attach that dataset to any later notebook the same way as the repo dataset.

**Download checklist:** the next cell lists every file under
`/kaggle/working/results/`. At minimum download `best_model.pt`,
`metrics.json`, `RESULTS.md`, `splits.json`, and the `baselines/`,
`explanations/` and `multimodal/` folders. Kaggle keeps `/kaggle/working`
only until the session ends, so download (or Save Version) before closing.


In [ ]:
# Cell 18: List every output file to download (path + size, relative to results/).
from pathlib import Path
results = Path('/kaggle/working/results')
if not results.exists():
    print("results/ does not exist - no cell above produced output yet. Run the notebook top to bottom first.")
else:
    for p in sorted(results.rglob('*')):
        if p.is_file():
            print(f"{p.relative_to(results)}  ({p.stat().st_size / 1024:.1f} KB)")
    print()
    print("Must-download: best_model.pt, metrics.json, RESULTS.md, splits.json,")
    print("  baselines/ (baseline_results.json + complexity report), explanations/, multimodal/.")